# 3 · Who reacts? Heterogeneous effects and uplift

The ATE hides heterogeneity. Meta-learners estimate the conditional effect τ(x) = E[Y(1) − Y(0) | X = x]; we rank customers by τ̂(x) and evaluate the ranking on a held-out set with Qini curves.
Propensity is known (0.5) because the data is randomized. Qini coefficients are normalized by the perfect ranking (treated responders first, control responders last).

**Segments** use the best model's τ̂(x) and the baseline P(Y | no e-mail): *Persuadables* τ̂ > δ, *Sleeping Dogs* τ̂ < −δ; otherwise *Sure Things* (high baseline) or *Lost Causes* (low baseline), with δ = ATE / 2. The segment table checks each group's uplift on held-out data.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from uplift_causal.data import load_hillstrom
from uplift_causal.evaluate_uplift import (
    budget_table,
    diff_means,
    qini_ci,
    qini_curve,
    segment_table,
    segments,
)
from uplift_causal.learners import baseline, fit_predict
from uplift_causal.plots import qini_plot, quadrant_plot, save

/Users/p.schmidt-petter/Desktop/dev/dev/uplift-causal-marketing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_hillstrom()
train, test = train_test_split(df, test_size=0.4, stratify=df.t * 2 + df.y, random_state=0)
X_train, X_test = train.drop(columns=["t", "y"]), test.drop(columns=["t", "y"])
models, tau = fit_predict(X_train, train.t, train.y, X_test)

In [3]:
qini = pd.DataFrame(
    {k: qini_ci(test.y, test.t, v) for k, v in tau.items()}, index=["qini", "lo", "hi"]
).T
best = qini.qini.idxmax()
qini.round(4)

,qini,lo,hi
T-Learner,0.0431,0.0149,0.0675
X-Learner,0.0427,0.0168,0.0676
Causal Forest,0.0630,0.0400,0.0878


In [4]:
pd.DataFrame(
    {"y": test.y.to_numpy(), "t": test.t.to_numpy()} | {k: v.astype("float32") for k, v in tau.items()}
).to_parquet("../app/scores_hillstrom.parquet", index=False)

In [5]:
save(qini_plot({k: qini_curve(test.y, test.t, v) for k, v in tau.items()}), "qini")

In [6]:
ate = diff_means(train.y, train.t)[0]
threshold, delta = train.y[train.t == 0].mean(), ate / 2
mu0 = baseline(models["T-Learner"], X_test)
seg = segments(mu0, tau[best], threshold, delta)
segment_table(test.y, test.t, seg).round(4)

/Users/p.schmidt-petter/Desktop/dev/dev/uplift-causal-marketing/src/uplift_causal/evaluate_uplift.py:12: RuntimeWarning: Degrees of freedom <= 0 for slice
  se = np.sqrt(y1.var(ddof=1) / len(y1) + y0.var(ddof=1) / len(y0))
/Users/p.schmidt-petter/Desktop/dev/dev/uplift-causal-marketing/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,share,uplift,lo,hi
Persuadables,0.6709,0.0609,0.0482,0.0736
Sure Things,0.1601,0.0025,-0.0227,0.0276
Lost Causes,0.1689,0.0221,0.0026,0.0416
Sleeping Dogs,0.0002,0.0000,NaN,NaN


In [7]:
save(quadrant_plot(mu0, tau[best], seg, threshold, delta), "segments", height=600)

In [8]:
budget = budget_table(test.y, test.t, tau[best])
row = budget.iloc[2]
print(
    f"{best}: targeting the top {row.cost_share:.0%} captures {row.effect_share:.0%} "
    f"(95% CI {row.lo:.0%}-{row.hi:.0%}) of the incremental visits."
)
budget.round(3)

Causal Forest: targeting the top 30% captures 53% (95% CI 43%-65%) of the incremental visits.


,cost_share,effect_share,lo,hi
0,0.1,0.144,0.077,0.219
1,0.2,0.366,0.271,0.485
2,0.3,0.526,0.428,0.654
3,0.5,0.800,0.695,0.932
4,0.7,0.911,0.829,1.020
5,1.0,1.000,1.000,1.000
